# Notebook 4 — Entrenamiento y evaluación (ConvNeXt-Tiny)

Entrena un clasificador binario **auténtica (0) / manipulada (1)** con el dataset que arma el notebook 3 y lo evalúa de la forma que pide la especificación: acierto por tipo de imagen, robustez a la compresión JPEG, mapa de calor (Grad-CAM) y semáforo.

**Decisión clave: el modelo mira recortes en resolución original, no la foto achicada.**
La zona editada ocupa entre el 3 % y el 8 % de la imagen. Si la foto de 768×512 se achica a 256×256, se borra justo el ruido fino que delata la edición. Por eso:

- **Entrenamiento:** recortes (*parches*) de 256×256 sin redimensionar. En las fotos editadas el recorte se ubica sobre la **zona editada** (usando la máscara del notebook 2); en la foto real "hermana", la mitad de los recortes cae en **la misma zona** sin editar y la otra mitad al azar. Así las dos clases muestran la misma parte del auto, y además el modelo ve fondos reales como ejemplo de "auténtico".

> **Corrida v1 (24/10/2026):** con el 100 % de los recortes de las reales sobre la zona editada, el modelo localizó bien las ediciones (Grad-CAM 6 veces más concentrado en la zona editada) pero marcó como sospechosas el 71 % de las fotos reales: casi nunca había visto fondos reales como "auténtico". De ahí `GUIADO_REALES = 0.5`.
- **Evaluación:** cada foto de 768×512 se divide en **6 recortes** (3 × 2) que cubren la imagen entera. La probabilidad de la foto es la del recorte más sospechoso: alcanza con que una zona esté editada.

**Antes de ejecutar:** haber corrido los notebooks 1, 2 y 3. Elegir *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

**Salida:** carpeta `04_modelo/` en Drive con el modelo entrenado (`mejor_modelo.pt`), la partición usada, las métricas y la configuración para el futuro tablero en Streamlit.

## 1. Instalación e imports

In [ ]:
import sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "albumentations", "grad-cam"], check=True)

In [ ]:
import os, io, json, time, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, confusion_matrix

## 2. Configuración

In [ ]:
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/deteccion-fraude-v2")
else:
    RAIZ = Path("./deteccion-fraude-v2")

FINAL         = RAIZ / "03_dataset_final"
MANIFEST      = FINAL / "manifest_final.csv"
DIR_MASCARAS  = RAIZ / "02_falsas" / "mascaras"
MANIFEST_FALS = RAIZ / "02_falsas" / "manifest_falsas.csv"     # se usa solo para saber qué modelo generó cada imagen
NOMBRE_CORRIDA = "v2"       # cada corrida guarda en su carpeta: 04_modelo_v2, 04_modelo_v3...
SALIDA        = RAIZ / f"04_modelo_{NOMBRE_CORRIDA}"
SALIDA.mkdir(parents=True, exist_ok=True)

PRUEBA_RAPIDA = False        # True: 2 épocas, para comprobar que todo corre antes de la corrida larga

W, H          = 768, 512     # tamaño de todas las imágenes del dataset
TAM_PARCHE    = 256          # recorte que ve el modelo (sin redimensionar)
PARCHES_POR_IMAGEN = 4       # recortes distintos de cada imagen por época
GUIADO_REALES = 0.5          # en las fotos reales, fracción de recortes sobre la zona editada de su hermana; el resto, al azar

NOMBRE_MODELO = "convnext_tiny.fb_in22k_ft_in1k"   # ConvNeXt-Tiny preentrenado (timm)
PREENTRENADO  = True
EPOCAS        = 2 if PRUEBA_RAPIDA else 20
PACIENCIA     = 5            # corta si la validación no mejora en esta cantidad de épocas
BATCH         = 16
LR, WD        = 1e-4, 1e-2
FRAC_VAL, FRAC_TEST = 0.15, 0.15
CORTE_AMARILLO, CORTE_ROJO = 0.40, 0.70      # semáforo: verde < 40 % <= amarillo < 70 % <= rojo
CAPA_GRADCAM  = -1           # -1: última etapa (mapa de 8×8 por recorte); -2: más detalle (16×16)
SEMILLA       = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEDIA, DESVIO = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])   # normalización ImageNet
torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)
print("Dispositivo:", DEVICE)
if DEVICE == "cpu":
    print("ATENCIÓN: no hay GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU T4.")

## 3. Lectura del dataset y de las máscaras

Para cada foto se busca la máscara de su edición (`<base_id>_mask.png`). La máscara la usan la foto editada **y su foto real de origen**: así el recorte de entrenamiento cae en la misma zona en las dos.

In [ ]:
df = pd.read_csv(MANIFEST)
df["ruta"] = [str(FINAL / a) for a in df["archivo"]]
faltan = [r for r in df["ruta"] if not Path(r).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan {len(faltan)} imágenes, por ejemplo {faltan[:3]}. ¿Corriste el notebook 3?")

def caja_de_mascara(base_id):
    """Rectángulo (x0, y0, x1, y1) que contiene la zona editada, o None si esa foto no tiene edición."""
    ruta = DIR_MASCARAS / f"{base_id}_mask.png"
    if not ruta.exists():
        return None
    m = Image.open(ruta).convert("L")
    if m.size != (W, H):
        m = m.resize((W, H), Image.NEAREST)
    ys, xs = np.nonzero(np.asarray(m) > 127)
    if len(xs) == 0:
        return None
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)

cajas = {b: caja_de_mascara(b) for b in df["base_id"].unique()}
df["caja"] = df["base_id"].map(cajas)
df.loc[df["tipo"].isin(["ia_sano", "ia_danio", "real_danado"]), "caja"] = None   # sin edición: recorte al azar

# qué modelo generativo hizo cada imagen manipulada (para medir el acierto por generador)
df["generador"] = np.where(df["clase"] == 0, "real", "desconocido")
if MANIFEST_FALS.exists():
    fals = pd.read_csv(MANIFEST_FALS)
    gen = {Path(a).stem: m.split("/")[-1] for a, m in zip(fals["archivo"], fals["modelo"])}
    df.loc[df["clase"] == 1, "generador"] = df.loc[df["clase"] == 1, "nombre"].map(gen).fillna("desconocido")

con_caja = df["caja"].notna()
print(f"{len(df)} imágenes | clase 0: {(df['clase'] == 0).sum()} | clase 1: {(df['clase'] == 1).sum()}")
print(f"Con zona de edición conocida (recorte guiado): {con_caja.sum()}")
sin_mascara = df[df["tipo"].str.startswith("inpaint") & ~con_caja]
if len(sin_mascara):
    print(f"ATENCIÓN: {len(sin_mascara)} ediciones sin máscara en {DIR_MASCARAS}; se recortarán al azar.")

## 4. Partición train / validación / test por foto base

Una foto real y sus ediciones comparten `base_id` y van **siempre a la misma partición**. Si la foto original quedara en entrenamiento y su versión editada en test, el modelo podría reconocer la foto en vez de detectar la edición.

In [ ]:
grupos = (df.groupby("base_id")["tipo"]
            .agg(lambda s: "+".join(sorted(set(s)))).rename("estrato").reset_index())

def dividir(g, frac):
    try:
        return train_test_split(g, test_size=frac, stratify=g["estrato"], random_state=SEMILLA)
    except ValueError:      # con muy pocas fotos no se puede estratificar
        return train_test_split(g, test_size=frac, random_state=SEMILLA)

g_resto, g_test = dividir(grupos, FRAC_TEST)
g_train, g_val  = dividir(g_resto, FRAC_VAL / (1 - FRAC_TEST))
particion = {**dict.fromkeys(g_train["base_id"], "train"),
             **dict.fromkeys(g_val["base_id"], "val"),
             **dict.fromkeys(g_test["base_id"], "test")}
df["particion"] = df["base_id"].map(particion)
assert df.groupby("base_id")["particion"].nunique().max() == 1

df.drop(columns=["ruta"]).to_csv(SALIDA / "particion.csv", index=False)
pd.crosstab([df["clase"], df["tipo"]], df["particion"], margins=True)[["train", "val", "test", "All"]]

## 5. Imágenes en memoria, recortes y aumentos

Las imágenes se leen una sola vez (leer de Drive en cada época es lento). Los **aumentos** imitan lo que le pasa a una foto real de siniestro: espejado, cambios suaves de luz y, sobre todo, **recompresión JPEG** (fotos reenviadas por WhatsApp). No se agregan desenfoques ni ruido artificial, porque borrarían o imitarían justo la señal que se busca.

In [ ]:
t0 = time.time()
CACHE = {}
for r in df["ruta"]:
    img = Image.open(r).convert("RGB")
    if img.size != (W, H):
        img = img.resize((W, H), Image.BICUBIC)
    CACHE[r] = np.asarray(img)
print(f"{len(CACHE)} imágenes cargadas en {time.time() - t0:.0f}s")

if int(A.__version__.split(".")[0]) >= 2:
    compresion = A.ImageCompression(quality_range=(40, 95), p=0.5)
else:
    compresion = A.ImageCompression(quality_lower=40, quality_upper=95, p=0.5)
AUMENTOS = A.Compose([A.HorizontalFlip(p=0.5),
                      A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
                      compresion])

def a_tensor(arr):
    x = (arr.astype(np.float32) / 255.0 - MEDIA) / DESVIO
    return torch.from_numpy(x.transpose(2, 0, 1).astype(np.float32))

def elegir_recorte(caja, rng):
    """Esquina superior izquierda del recorte. Con caja: el recorte contiene parte de la zona editada."""
    if caja is not None:
        x0, y0, x1, y1 = caja
        cx, cy = rng.integers(x0, x1), rng.integers(y0, y1)        # un punto de la zona editada
        x = cx - rng.integers(32, TAM_PARCHE - 32)
        y = cy - rng.integers(32, TAM_PARCHE - 32)
    else:
        x, y = rng.integers(0, W - TAM_PARCHE + 1), rng.integers(0, H - TAM_PARCHE + 1)
    return int(np.clip(x, 0, W - TAM_PARCHE)), int(np.clip(y, 0, H - TAM_PARCHE))

class Parches(Dataset):
    def __init__(self, tabla):
        self.t = tabla.reset_index(drop=True)
        self.rng = np.random.default_rng(SEMILLA)
    def __len__(self):
        return len(self.t) * PARCHES_POR_IMAGEN
    def __getitem__(self, i):
        f = self.t.iloc[i % len(self.t)]
        caja = f["caja"]
        # En las reales, parte de los recortes va a la zona de la edición (misma zona que su hermana editada)
        # y el resto al azar, para que el modelo también vea fondo, piso y cielo reales como "auténtico".
        if f["clase"] == 0 and caja is not None and self.rng.random() >= GUIADO_REALES:
            caja = None
        x, y = elegir_recorte(caja, self.rng)
        parche = CACHE[f["ruta"]][y:y + TAM_PARCHE, x:x + TAM_PARCHE]
        parche = AUMENTOS(image=np.ascontiguousarray(parche))["image"]
        return a_tensor(parche), int(f["clase"])

def recortes_grilla(arr):
    """Los 6 recortes (2 filas × 3 columnas) que cubren una imagen de 768×512."""
    return [arr[y:y + TAM_PARCHE, x:x + TAM_PARCHE]
            for y in range(0, H, TAM_PARCHE) for x in range(0, W, TAM_PARCHE)]

train_df = df[df["particion"] == "train"]
val_df   = df[df["particion"] == "val"]
test_df  = df[df["particion"] == "test"]
carga_train = DataLoader(Parches(train_df), batch_size=BATCH, shuffle=True, num_workers=0, drop_last=False)

# vista rápida de algunos recortes de entrenamiento
muestra = [Parches(train_df)[i] for i in range(0, min(len(train_df), 8))]
fig, ax = plt.subplots(1, len(muestra), figsize=(2.2 * len(muestra), 2.6))
for a, (t, c) in zip(np.atleast_1d(ax), muestra):
    a.imshow(np.clip(t.numpy().transpose(1, 2, 0) * DESVIO + MEDIA, 0, 1)); a.set_title(f"clase {c}"); a.axis("off")
plt.tight_layout(); plt.show()

## 6. Modelo y predicción por foto completa

ConvNeXt-Tiny preentrenado en ImageNet, con una salida de 2 clases. La **entropía cruzada con pesos por clase** compensa si una clase tiene más imágenes que la otra.

In [ ]:
modelo = timm.create_model(NOMBRE_MODELO, pretrained=PREENTRENADO, num_classes=2).to(DEVICE)

n_clase = train_df["clase"].value_counts().reindex([0, 1], fill_value=0).values.astype(float)
pesos = torch.tensor(n_clase.sum() / (2 * np.maximum(n_clase, 1)), dtype=torch.float32, device=DEVICE)
criterio = nn.CrossEntropyLoss(weight=pesos)
optimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=WD)
planificador = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=EPOCAS)
usar_amp = DEVICE == "cuda"
escalador = torch.amp.GradScaler("cuda", enabled=usar_amp)
print(f"Pesos por clase: {pesos.cpu().numpy().round(2)}")

@torch.no_grad()
def prob_recortes(arrs, m=None):
    """Probabilidad de 'manipulada' de cada recorte de cada imagen: matriz (n_imágenes, 6)."""
    m = m or modelo
    m.eval()
    salida = []
    for arr in arrs:
        x = torch.stack([a_tensor(r) for r in recortes_grilla(arr)]).to(DEVICE)
        with torch.autocast(device_type="cuda", enabled=usar_amp):
            salida.append(torch.softmax(m(x).float(), dim=1)[:, 1].cpu().numpy())
    return np.array(salida)

def prob_imagenes(tabla, m=None, transformar=None):
    """Probabilidad por foto = la del recorte más sospechoso."""
    arrs = [CACHE[r] if transformar is None else transformar(CACHE[r]) for r in tabla["ruta"]]
    return prob_recortes(arrs, m).max(axis=1)

def auc_seguro(y, p):
    return roc_auc_score(y, p) if len(set(y)) == 2 else float("nan")

## 7. Entrenamiento

En cada época se mide el **AUC en validación** sobre las fotos completas y se guarda el mejor modelo. Si la validación no mejora durante `PACIENCIA` épocas, se corta.

In [ ]:
historial, mejor_auc, sin_mejora = [], -1.0, 0
ruta_modelo = SALIDA / "mejor_modelo.pt"
for epoca in range(1, EPOCAS + 1):
    t0 = time.time()
    modelo.train()
    perdida_total, n = 0.0, 0
    for x, y in carga_train:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizador.zero_grad()
        with torch.autocast(device_type="cuda", enabled=usar_amp):
            perdida = criterio(modelo(x), y)
        escalador.scale(perdida).backward()
        escalador.step(optimizador); escalador.update()
        perdida_total += perdida.item() * len(y); n += len(y)
    planificador.step()

    p_val = prob_imagenes(val_df)
    y_val = val_df["clase"].values
    auc_val = auc_seguro(y_val, p_val)
    acc_val = float(((p_val >= 0.5) == y_val).mean()) if len(y_val) else float("nan")
    historial.append({"epoca": epoca, "perdida_train": perdida_total / max(n, 1), "auc_val": auc_val, "acierto_val": acc_val})
    criterio_val = auc_val if not np.isnan(auc_val) else acc_val
    marca = ""
    if criterio_val > mejor_auc:
        mejor_auc, sin_mejora, marca = criterio_val, 0, "  <- mejor, guardado"
        torch.save(modelo.state_dict(), ruta_modelo)
    else:
        sin_mejora += 1
    print(f"Época {epoca:2d} | pérdida {historial[-1]['perdida_train']:.3f} | AUC val {auc_val:.3f} | "
          f"acierto val {acc_val:.2f} | {time.time() - t0:.0f}s{marca}")
    if sin_mejora >= PACIENCIA:
        print("La validación no mejora: se corta el entrenamiento.")
        break

hist = pd.DataFrame(historial)
hist.to_csv(SALIDA / "historial.csv", index=False)
modelo.load_state_dict(torch.load(ruta_modelo, map_location=DEVICE))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist["epoca"], hist["perdida_train"], marker="o"); ax[0].set_title("Pérdida en entrenamiento")
ax[1].plot(hist["epoca"], hist["auc_val"], marker="o", label="AUC")
ax[1].plot(hist["epoca"], hist["acierto_val"], marker="o", label="acierto"); ax[1].set_title("Validación"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Evaluación en test

Las fotos de test **no se usaron** ni para entrenar ni para elegir el modelo.

- **AUC**: qué tan bien ordena el modelo las fotos (1 = perfecto, 0,5 = al azar).
- **Recall de manipuladas**: de las fotos manipuladas, qué porcentaje detecta. En un sistema de triaje es la métrica más importante: un fraude no detectado sigue de largo.
- **Precisión**: de las fotos que marca, qué porcentaje está realmente manipulado. Si es baja, los peritos revisan muchas fotos auténticas.
- **Acierto por tipo**: la prueba más exigente es `inpaint_control` (edición sin daño). Si el modelo acierta mucho en las 100 % IA y poco en las ediciones, aprendió la diferencia más fácil.

In [ ]:
def semaforo(p):
    return "rojo" if p >= CORTE_ROJO else ("amarillo" if p >= CORTE_AMARILLO else "verde")

test = test_df.copy()
test["prob"] = prob_imagenes(test)
test["pred"] = (test["prob"] >= 0.5).astype(int)
test["semaforo"] = test["prob"].map(semaforo)
y, p = test["clase"].values, test["prob"].values

metricas = {
    "n_test": int(len(test)),
    "auc": auc_seguro(y, p),
    "auc_pr": average_precision_score(y, p) if len(set(y)) == 2 else float("nan"),
    "recall_manipuladas": recall_score(y, test["pred"], zero_division=0),
    "precision_manipuladas": precision_score(y, test["pred"], zero_division=0),
    "acierto": float((test["pred"] == y).mean()),
}
for k, v in metricas.items():
    print(f"{k:24s} {v:.3f}" if isinstance(v, float) else f"{k:24s} {v}")
print("\nMatriz de confusión (filas: real, columnas: predicha)\n", confusion_matrix(y, test["pred"], labels=[0, 1]))

por_tipo = (test.assign(acierto=(test["pred"] == test["clase"]))
                .groupby(["clase", "tipo"]).agg(n=("acierto", "size"), acierto=("acierto", "mean"),
                                                prob_media=("prob", "mean")).round(2))
print("\nAcierto por tipo:"); display(por_tipo)
print("Acierto por generador:")
display(test.assign(acierto=(test["pred"] == test["clase"])).groupby("generador")["acierto"].agg(["size", "mean"]).round(2))
print("Semáforo por tipo (cantidad de fotos):")
display(pd.crosstab(test["tipo"], test["semaforo"]).reindex(columns=["verde", "amarillo", "rojo"], fill_value=0))
test.drop(columns=["ruta"]).to_csv(SALIDA / "predicciones_test.csv", index=False)

## 9. Robustez a la compresión JPEG

Las fotos de un siniestro suelen llegar recomprimidas (WhatsApp, mail). Se vuelve a comprimir **todo** el test (las dos clases) con calidades cada vez más bajas y se mide cuánto cae el modelo. Una caída fuerte indica que depende de detalles muy finos que la compresión borra.

In [ ]:
def recomprimir(calidad):
    def f(arr):
        buf = io.BytesIO()
        Image.fromarray(arr).save(buf, "JPEG", quality=calidad)
        return np.asarray(Image.open(buf).convert("RGB"))
    return f

filas = [{"calidad_jpeg": "original", "auc": metricas["auc"], "recall_manipuladas": metricas["recall_manipuladas"]}]
for q in [90, 70, 50, 30]:
    pq = prob_imagenes(test, transformar=recomprimir(q))
    filas.append({"calidad_jpeg": q, "auc": auc_seguro(y, pq),
                  "recall_manipuladas": recall_score(y, (pq >= 0.5).astype(int), zero_division=0)})
robustez = pd.DataFrame(filas).round(3)
robustez

## 10. Mapa de calor (Grad-CAM) y control contra las máscaras

Grad-CAM muestra **qué zona de la foto** llevó al modelo a sospechar. Se calcula en cada uno de los 6 recortes y se arma el mapa de la foto completa; cada recorte se pondera por su probabilidad, para que un recorte limpio no se vea "caliente".

Como de las fotos editadas se conoce la máscara real de la edición, se puede comprobar si el modelo mira el lugar correcto:
- **Puntería:** el punto más caliente del mapa cae dentro de la zona editada (con un margen de 16 px).
- **Calor en la zona:** qué parte del calor total cae en la zona editada, comparado con el tamaño de la zona. Un valor mayor que 1 indica que el modelo se concentra en la edición.

In [ ]:
capa_cam = [modelo.stages[CAPA_GRADCAM]]

def mapa_calor(arr):
    """Mapa de calor 512×768 (0 a 1) y probabilidad por recorte."""
    recortes = recortes_grilla(arr)
    x = torch.stack([a_tensor(r) for r in recortes]).to(DEVICE)
    probs = prob_recortes([arr])[0]
    modelo.eval()
    with GradCAM(model=modelo, target_layers=capa_cam) as cam:
        mapas = cam(input_tensor=x, targets=[ClassifierOutputTarget(1)] * len(recortes))
    mapa = np.zeros((H, W), dtype=np.float32)
    k = 0
    for yy in range(0, H, TAM_PARCHE):
        for xx in range(0, W, TAM_PARCHE):
            mapa[yy:yy + TAM_PARCHE, xx:xx + TAM_PARCHE] = mapas[k] * probs[k]
            k += 1
    return mapa, probs

def mascara_de(base_id):
    ruta = DIR_MASCARAS / f"{base_id}_mask.png"
    if not ruta.exists():
        return None
    m = Image.open(ruta).convert("L").resize((W, H), Image.NEAREST)
    return np.asarray(m) > 127

def dilatar(m, px=16):
    import cv2
    return cv2.dilate(m.astype(np.uint8), np.ones((2 * px + 1, 2 * px + 1), np.uint8)) > 0

ediciones = test[test["tipo"].str.startswith("inpaint")]
control = []
for _, f in ediciones.iterrows():
    m = mascara_de(f["base_id"])
    if m is None or not m.any():
        continue
    mapa, _ = mapa_calor(CACHE[f["ruta"]])
    yy, xx = np.unravel_index(mapa.argmax(), mapa.shape)
    total = mapa.sum()
    control.append({"nombre": f["nombre"], "tipo": f["tipo"], "prob": f["prob"],
                    "punteria": bool(dilatar(m)[yy, xx]),
                    "calor_en_zona": (mapa[m].sum() / total) / m.mean() if total > 0 else float("nan")})
control = pd.DataFrame(control)
if len(control):
    display(control.groupby("tipo")[["punteria", "calor_en_zona"]].mean().round(2))
else:
    print("No hay ediciones con máscara en test.")

In [ ]:
def mostrar(f, ax_foto, ax_mapa):
    arr = CACHE[f["ruta"]]
    mapa, _ = mapa_calor(arr)
    ax_foto.imshow(arr); ax_foto.set_title(f"{f['tipo']} | P={f['prob']:.2f} ({semaforo(f['prob'])})", fontsize=9)
    ax_mapa.imshow(arr); ax_mapa.imshow(mapa, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    m = mascara_de(f["base_id"]) if f["tipo"].startswith("inpaint") else None
    if m is not None:
        ax_mapa.contour(m, levels=[0.5], colors="white", linewidths=1)
    ax_mapa.set_title("Grad-CAM (blanco: zona editada real)", fontsize=9)
    ax_foto.axis("off"); ax_mapa.axis("off")

ejemplos = pd.concat([g.head(1) for _, g in test.groupby("tipo")])
fig, ax = plt.subplots(len(ejemplos), 2, figsize=(10, 3.4 * len(ejemplos)), squeeze=False)
for i, (_, f) in enumerate(ejemplos.iterrows()):
    mostrar(f, ax[i, 0], ax[i, 1])
plt.tight_layout(); plt.show()

## 11. Analizar una foto nueva

`analizar_foto(ruta)` recibe cualquier foto, la lleva al formato del dataset (768×512, recorte central 3:2) y devuelve la probabilidad de manipulación, el color del semáforo y el mapa de calor. Es la función que va a usar el tablero de Streamlit.

**Ojo:** si la foto tiene otro tamaño, el reescalado modifica el ruido fino. Para fotos de celular de alta resolución, los resultados son orientativos hasta probar con más fotos de ese tipo.

In [ ]:
def preparar(ruta):
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    if img.size != (W, H):
        img = ImageOps.fit(img, (W, H), Image.BICUBIC)     # escala y recorta al centro en proporción 3:2
    return np.asarray(img)

def analizar_foto(ruta, mostrar_figura=True):
    arr = preparar(ruta)
    mapa, probs = mapa_calor(arr)
    p = float(probs.max())
    color = semaforo(p)
    accion = {"verde": "sigue el circuito habitual",
              "amarillo": "revisión de un perito con el mapa de calor",
              "rojo": "peritaje presencial o videoperitaje"}[color]
    if mostrar_figura:
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
        ax[0].imshow(arr); ax[0].axis("off"); ax[0].set_title(Path(ruta).name)
        ax[1].imshow(arr); ax[1].imshow(mapa, cmap="jet", alpha=0.45, vmin=0, vmax=1); ax[1].axis("off")
        ax[1].set_title(f"P(manipulada) = {p:.0%} → {color.upper()}: {accion}")
        plt.tight_layout(); plt.show()
    return {"prob_manipulada": p, "semaforo": color, "accion": accion, "mapa": mapa}

# ejemplo con una foto del test:
_ = analizar_foto(test.iloc[0]["ruta"])

## 12. Guardar configuración y resultados

In [ ]:
config = {"nombre_modelo": NOMBRE_MODELO, "archivo_pesos": "mejor_modelo.pt", "ancho": W, "alto": H,
          "tam_parche": TAM_PARCHE, "agregacion": "max", "media": MEDIA.tolist(), "desvio": DESVIO.tolist(),
          "corte_amarillo": CORTE_AMARILLO, "corte_rojo": CORTE_ROJO, "capa_gradcam": CAPA_GRADCAM}
(SALIDA / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
resultados = {"metricas_test": metricas, "robustez_jpeg": robustez.to_dict("records"),
              "acierto_por_tipo": por_tipo.reset_index().to_dict("records"),
              "gradcam": control.groupby("tipo")[["punteria", "calor_en_zona"]].mean().round(3).reset_index().to_dict("records") if len(control) else []}
(SALIDA / "resultados.json").write_text(json.dumps(resultados, indent=2, ensure_ascii=False, default=float))
print("Guardado en", SALIDA)
for a in sorted(SALIDA.iterdir()):
    print(" ", a.name)

## 13. Cómo leer los resultados

- **Con pocos cientos de fotos el test tiene pocas imágenes por tipo**: un acierto del 100 % o del 50 % puede cambiar mucho con unas pocas fotos más. Sirve para comparar variantes del modelo, no como desempeño final.
- Si `inpaint_control` tiene acierto bajo y `inpaint_danio` alto, el modelo está usando el **daño** como pista y no la edición.
- Si el acierto en las 100 % IA es alto y en las ediciones bajo, aprendió la diferencia más fácil.
- Si la **puntería** del Grad-CAM es baja aunque el acierto sea alto, el modelo acierta por algo distinto de la zona editada: hay que revisar.
- Los cortes del semáforo (40 / 70 %) son iniciales: conviene ajustarlos con la validación según cuántas fotos pueden revisar los peritos.